# GameTheory-16e : LLM hétérogènes comme joueurs — Othman–Sandholm (pilote)

**Navigation** : [<< 16-MechanismDesign](GameTheory-16-MechanismDesign.ipynb) | [Index](README.md) | [17-MultiAgent-RL >>](GameTheory-17-MultiAgent-RL.ipynb)

**Side track** : pilote qui branche de **vrais LLM hétérogènes** comme joueurs sur le mécanisme Othman–Sandholm (Proposition 6, SAGT 2009) déjà exécutable dans `GameTheory-16`. Deux familles LLM réelles (via proxy claudish), deux baselines scriptées, replay déterministe par l'oracle du mécanisme.

**Voir** : Issue #15399, Epic #15397, PR Lean #12343 (Proposition 6 formalisée par `decide`).

## Pré-enregistrement (H0/H1, avant le premier appel LLM)

**Mécanisme owner** : `GameTheory-16-MechanismDesign.ipynb` §4.6 — Proposition 6 d'Othman–Sandholm (SAGT 2009). 2 agents *row* et *column*, chacun a 2 types (`a` ou `a'`), 4 profils de rapport → 4 outcomes `o1`..`o4`. Caractéristique 1 (Lean #12343) : rapporter `a'` est strictement dominant pour les deux agents. Caractéristique 2 : dès qu'un agent dévie, il existe un outcome avec `SW > SW(o1)` sauf pour le profil `(a', a')`.

**Mécanique vérifiée firsthand** : `u_row`/`u_col` (table 4×2), `outcomes = ["o1","o2","o3","o4"]`, oracle déterministe `mom_outcome = "o1"` (boxed truthful), `sw_at_mom[o, tr, tc]` calculé verbatim comme dans cell 28 du notebook owner.

**Hypothèses testées** :
- **H0** : un LLM *sans* visibilité du code/du tableau de gains obtient un payoff moyen **<=** baselines scriptées (conformiste DSIC et byzantin uniforme), sur les 4 profils.
- **H1** : au moins une famille LLM *avec* visibilité du code/de la table obtient un payoff moyen **>** baseline conformiste et une manipulabilité **<** baseline byzantine.
- **Verdict négatif possible** : H0 et H1 peuvent être infirmés — le protocole accepte les deux issues. Le pilote documente la mesure, pas la conclusion attendue.

**Conditions minimales du pilote** (cf. #15399) :
1. Politiques scriptées — conformiste DSIC + byzantin uniforme — sur les mêmes instances.
2. Au moins deux familles réelles de LLM, hétérogènes (pas le même backend).
3. LLM **avec** règles/code visibles (condition « voit ») vs LLM **sans** (condition « aveugle »).
4. Scénario hostile : permutation d'une règle + coût neutralisé pour vérifier que l'avantage LLM disparaît.
5. ≥4 seeds (0/1/7/42) — stochastique côté LLM.

**Métriques mesurées** : payoff par rôle, bien-être social `SW`, revenu mécanicien `M(o) = u_row(o|tr) + u_col(o|tc)`, violations DSIC, manipulabilité (gain par rapport à truthful), coût USD (tokens), latence ms/decision, refus/parse-error.

**Pas de claim Löb/FairBot fort** (cf. #15062/#15066) — le pilote porte la **manipulabilité**, pas la vérité gödelienne.


## 1. Imports et configuration

L'environnement n'embarque pas de secrets. Le notebook charge `ANTHROPIC_API_KEY` depuis `.secrets/master.env` (gitignored) si présent, sinon lève une erreur explicite — pas de fallback littéral (cf. `secrets-hygiene.md` règle 2). Le proxy claudish est sur `ANTHROPIC_BASE_URL` (défaut `http://192.168.0.46:3000`). Les appels LLM passent par `/v1/chat/completions` avec `Authorization: Bearer <key>`.


In [1]:
import os
import json
import time
import statistics
from pathlib import Path
from typing import Dict, List, Tuple, Any

import requests

# Constante pour le type/rapport a' : assemblage pour éviter les conflits d'apostrophe.
A_PRIME = 'a' + chr(39)  # 'a' followed by ASCII apostrophe

# Charger les secrets gitignored (NE PAS hardcoder de fallback littéral -- secrets-hygiene.md R2)
_master_env = Path('.secrets/master.env')
if _master_env.exists():
    for line in _master_env.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

PROXY_BASE = os.environ.get('ANTHROPIC_BASE_URL', 'http://192.168.0.46:3000').rstrip('/')
API_KEY = os.environ.get('ANTHROPIC_API_KEY')

if not API_KEY:
    raise RuntimeError(
        'ANTHROPIC_API_KEY absent. Configurer .secrets/master.env (gitignored) '
        "ou exposer la variable d'environnement. JAMAIS de fallback littéral inline."
    )

print(f'Proxy claudish: {PROXY_BASE}')
print('API key chargée: ' + '*' * 8 + API_KEY[-4:])

SEEDS = [0, 1, 7, 42]
TYPES = ['a', A_PRIME]


Proxy claudish: http://192.168.0.46:3000
API key chargée: ********EAAA


## 2. Oracle Othman–Sandholm (copié verbatim du notebook 16 §4.6)

L'oracle retourne le payoff par rôle et le bien-être social pour un profil de types vrais et un profil de rapports. **Strict MOM** : si tous les agents rapportent sincèrement (rapport = type), le mécanisme retourne `o1` (boxed truthful). Si un agent dévie, l'oracle révèle l'outcome effectif avec son `SW`.


In [2]:
# Table de gains (Othman-Sandholm 2009, Proposition 6) -- copiée verbatim de
# GameTheory-16-MechanismDesign.ipynb cell 28.
U_ROW = {
    'o1': {'a': 1, A_PRIME: 3},
    'o2': {'a': 4, A_PRIME: 5},
    'o3': {'a': 0, A_PRIME: 0},
    'o4': {'a': 3, A_PRIME: 0},
}
U_COL = {
    'o1': {'a': 1, A_PRIME: 4},
    'o2': {'a': 0, A_PRIME: 0},
    'o3': {'a': 3, A_PRIME: 6},
    'o4': {'a': 0, A_PRIME: 0},
}
OUTCOMES = ['o1', 'o2', 'o3', 'o4']
MOM_OUTCOME = 'o1'


def sw(outcome, type_row, type_col):
    """Bien-être social : somme des utilités des deux agents pour cet outcome et profil de types."""
    return U_ROW[outcome][type_row] + U_COL[outcome][type_col]


def payoff_row(outcome, type_row):
    return U_ROW[outcome][type_row]


def payoff_col(outcome, type_col):
    return U_COL[outcome][type_col]


def outcome_strict_mom(report_row, report_col):
    """
    Mécanisme M_1 boxed truthful : retourne o1 si (row=a', col=a') -- profil sincère canonique.
    Sinon, retourne l'outcome optimal en SW pour le profil de RAPPORTS (pas de types vrais).
    La formalisation Lean (PR #12343) prouve la Caractéristique 2 sur le profil de TYPES.
    Ici, l'oracle retourne l'outcome pour le profil de RAPPORTS.
    """
    if report_row == A_PRIME and report_col == A_PRIME:
        return MOM_OUTCOME
    return max(OUTCOMES, key=lambda o: U_ROW[o][report_row] + U_COL[o][report_col])


# Vérification -- identique au notebook 16 cell 28
print('SW(o | tr, tc) pour les 4 outcomes × 4 profils de types :')
print('{:<8}'.format('Outcome'), end='')
for tr, tc in [(tr, tc) for tr in TYPES for tc in TYPES]:
    print('  ({},{}) '.format(tr, tc), end='')
print()
for o in OUTCOMES:
    row = '{:<8}'.format(o)
    for tr, tc in [(tr, tc) for tr in TYPES for tc in TYPES]:
        row += '  {:>5}   '.format(sw(o, tr, tc))
    print(row)

print('\nCaractéristique 2 (strict MOM) -- pour chaque profil, outcome non-M1 avec SW > SW(o1) :')
for tr, tc in [(tr, tc) for tr in TYPES for tc in TYPES]:
    sw_o1 = sw(MOM_OUTCOME, tr, tc)
    better = [o for o in OUTCOMES if sw(o, tr, tc) > sw_o1]
    print('  ({},{}) : SW(o1)={}, outcomes strictement meilleurs : {}'.format(tr, tc, sw_o1, better))


SW(o | tr, tc) pour les 4 outcomes × 4 profils de types :
Outcome   (a,a)   (a,a')   (a',a)   (a',a') 
o1            2         5         4         7   
o2            4         4         5         5   
o3            3         6         3         6   
o4            3         3         0         0   

Caractéristique 2 (strict MOM) -- pour chaque profil, outcome non-M1 avec SW > SW(o1) :
  (a,a) : SW(o1)=2, outcomes strictement meilleurs : ['o2', 'o3', 'o4']
  (a,a') : SW(o1)=5, outcomes strictement meilleurs : ['o3']
  (a',a) : SW(o1)=4, outcomes strictement meilleurs : ['o2']
  (a',a') : SW(o1)=7, outcomes strictement meilleurs : []


## 3. Baselines scriptées (2 politiques déterministes)

**Conformiste DSIC** : rapporte son type vrai en permanence. Maximise le payoff quand l'autre fait pareil (par la dominance stricte de `a'`, cf. PR #12343). Aucune déviation.

**Byzantin uniforme** : rapporte `a` quel que soit son type. Stress-test : voir comment l'oracle et les autres politiques se comportent face à une déviation constante.


In [3]:
def baseline_conformist(true_type, opponent_report):
    """DSIC : rapporte son type vrai. Pas de seed, pas d'aléa."""
    return true_type


def baseline_byzantine(true_type, opponent_report):
    """Byzantin uniforme : rapporte toujours 'a'."""
    return 'a'


BASELINES = {
    'conformist': baseline_conformist,
    'byzantine': baseline_byzantine,
}


def run_baseline_game(policy_row, policy_col, type_row, type_col):
    """Joue une partie entre deux politiques (row/col) avec types vrais."""
    rep_row = policy_row(type_row, None)
    rep_col = policy_col(type_col, None)
    outcome = outcome_strict_mom(rep_row, rep_col)
    return {
        'type_row': type_row,
        'type_col': type_col,
        'report_row': rep_row,
        'report_col': rep_col,
        'outcome': outcome,
        'payoff_row': payoff_row(outcome, type_row),
        'payoff_col': payoff_col(outcome, type_col),
        'sw': sw(outcome, type_row, type_col),
    }


print("Payoffs baselines (4 profils × 2 politiques × 2 rôles) :\n")
for pol_name in BASELINES:
    print('--- {} ---'.format(pol_name))
    for tr, tc in [(tr, tc) for tr in TYPES for tc in TYPES]:
        if pol_name == 'conformist':
            res = run_baseline_game(BASELINES['conformist'], BASELINES['conformist'], tr, tc)
        else:
            res = run_baseline_game(BASELINES['byzantine'], BASELINES['byzantine'], tr, tc)
        print('  ({},{}) : rep=({},{}) outcome={} u_row={} u_col={} SW={}'.format(
            tr, tc, res['report_row'], res['report_col'], res['outcome'],
            res['payoff_row'], res['payoff_col'], res['sw']))


Payoffs baselines (4 profils × 2 politiques × 2 rôles) :

--- conformist ---
  (a,a) : rep=(a,a) outcome=o2 u_row=4 u_col=0 SW=4
  (a,a') : rep=(a,a') outcome=o3 u_row=0 u_col=6 SW=6
  (a',a) : rep=(a',a) outcome=o2 u_row=5 u_col=0 SW=5
  (a',a') : rep=(a',a') outcome=o1 u_row=3 u_col=4 SW=7
--- byzantine ---
  (a,a) : rep=(a,a) outcome=o2 u_row=4 u_col=0 SW=4
  (a,a') : rep=(a,a) outcome=o2 u_row=4 u_col=0 SW=4
  (a',a) : rep=(a,a) outcome=o2 u_row=5 u_col=0 SW=5
  (a',a') : rep=(a,a) outcome=o2 u_row=5 u_col=0 SW=5


## 4. Joueurs LLM hétérogènes

Le proxy claudish expose 21 modèles sous des noms *alias*. Les noms `claude-haiku-4-5` et `claude-sonnet-5` sont mappés en interne à **deux backends réellement hétérogènes** (vérifié ci-dessous par le champ `model` dans la réponse), via le même endpoint `/v1/chat/completions`. C'est l'hétérogénéité effective dont le pilote a besoin — deux moteurs distincts, deux latences, deux familles de tokens.

**Conditions** :
- **« voit »** : le prompt inclut le code Python de l'oracle + la table de gains `U_ROW`/`U_COL`. Le LLM a l'information complète.
- **« aveugle »** : le prompt ne contient que la règle du jeu en français, sans table ni code. Le LLM doit deviner ou négocier.


In [4]:
ORACLE_CODE_VISIBLE = (
    'Le mécanisme M est le suivant (code Python) :\n'
    '    U_ROW = {\'o1\': {\'a\': 1, \'a' + chr(39) + '\': 3}, \'o2\': {\'a\': 4, \'a' + chr(39) + '\': 5},\n'
    '             \'o3\': {\'a\': 0, \'a' + chr(39) + '\': 0}, \'o4\': {\'a\': 3, \'a' + chr(39) + '\': 0}}\n'
    '    U_COL = {\'o1\': {\'a\': 1, \'a' + chr(39) + '\': 4}, \'o2\': {\'a\': 0, \'a' + chr(39) + '\': 0},\n'
    '             \'o3\': {\'a\': 3, \'a' + chr(39) + '\': 6}, \'o4\': {\'a\': 0, \'a' + chr(39) + '\': 0}}\n'
    '    def outcome(rep_r, rep_c):\n'
    '        if rep_r == \'a' + chr(39) + '\' and rep_c == \'a' + chr(39) + '\': return \'o1\'\n'
    '        return max([\'o1\',\'o2\',\'o3\',\'o4\'],\n'
    '                   key=lambda o: U_ROW[o][rep_r] + U_COL[o][rep_c])\n'
    '    def payoff_row(o, tr): return U_ROW[o][tr]\n'
    '    def payoff_col(o, tc): return U_COL[o][tc]\n'
)

PROMPT_VISIBLE_TEMPLATE = (
    "Tu joues au jeu Othman-Sandholm contre un autre joueur.\n"
    "Règles en langage naturel :\n"
    "- Tu es un des deux joueurs. Tu as un type privé parmi {{a, a_prime}}. Tu ne connais PAS le type de l'autre.\n"
    "- Tu choisis un rapport parmi {{a, a_prime}} (un message que tu envoies au mécanisme).\n"
    "- Le mécanisme produit un outcome o1, o2, o3 ou o4. Ton utility dépend de l'outcome ET de TON type.\n"
    "\n"
    "{oracle_code}\n"
    "\n"
    "Ton type (privé) : {type_self}\n"
    "Le rapport de l'adversaire est : {opponent_report}\n"
    "Réponds UNIQUEMENT par 'a' ou par \"a'\" (avec l'apostrophe). Aucun autre texte.\n"
)

PROMPT_BLIND_TEMPLATE = (
    "Tu joues à un jeu à deux joueurs. Tu as un type privé parmi {{a, a_prime}}. Tu choisis un rapport parmi {{a, a_prime}}. "
    "Le mécanisme retourne un outcome parmi o1..o4. Ton utility dépend de l'outcome et de ton type.\n"
    "\n"
    "Ton type (privé) : {type_self}\n"
    "Le rapport de l'adversaire est : {opponent_report}\n"
    "\n"
    "Réponds UNIQUEMENT par 'a' ou par \"a'\" (avec l'apostrophe). Aucun autre texte, aucune explication.\n"
)


def parse_report(text):
    """Robust parse : accepte 'a', 'a' + apostrophe, 'A', etc."""
    t = text.strip().strip('"').strip(chr(39)).lower()
    if (A_PRIME in t) or ('apostrophe' in t):
        return A_PRIME
    if t.startswith('a'):
        return 'a'
    # refus / parse error : défaut conservateur = 'a'
    return 'a'


def call_llm(model_alias, prompt, temperature=0.7, seed=0, max_retries=2, timeout=30):
    """Appel réel via le proxy claudish. Retourne dict avec report + métriques."""
    body = {
        'model': model_alias,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': 8,
        'temperature': temperature,
        'seed': seed,
    }
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer ' + API_KEY,
    }
    last_err = None
    for attempt in range(max_retries + 1):
        t0 = time.time()
        try:
            r = requests.post(
                PROXY_BASE + '/v1/chat/completions',
                headers=headers, json=body, timeout=timeout,
            )
            dt_ms = int((time.time() - t0) * 1000)
            if r.status_code == 200:
                j = r.json()
                content = j['choices'][0]['message']['content'] or ''
                if not content.strip():
                    rc = j['choices'][0]['message'].get('reasoning_content')
                    content = rc or content
                return {
                    'report': parse_report(content),
                    'raw_content': content[:100],
                    'model_effective': j.get('model', model_alias),
                    'latency_ms': dt_ms,
                    'usage': j.get('usage', {}),
                    'status': 200,
                    'attempt': attempt + 1,
                }
            last_err = 'HTTP {}: {}'.format(r.status_code, r.text[:120])
        except Exception as e:
            dt_ms = int((time.time() - t0) * 1000)
            last_err = 'Exception: {!r}'.format(e)
        if attempt < max_retries:
            time.sleep(0.5)
    return {
        'report': 'a',
        'raw_content': '',
        'model_effective': model_alias,
        'latency_ms': -1,
        'usage': {},
        'status': -1,
        'error': last_err,
        'attempt': max_retries + 1,
    }


# Test ping : vérifier que les 2 familles hétérogènes répondent (HARD)
print('=== PING 2 familles hétérogènes (1ère mesure) ===')
for alias in ['claude-haiku-4-5', 'claude-sonnet-5']:
    res = call_llm(alias, 'Reply with ONLY the letter A.', temperature=0.0, seed=0)
    print('  {:20} -> HTTP {} {:>5} ms model_eff={:14} report={!r}'.format(
        alias, res['status'], res['latency_ms'], res['model_effective'], res['report']))


=== PING 2 familles hétérogènes (1ère mesure) ===


  claude-haiku-4-5     -> HTTP 200  3206 ms model_eff=MiniMax-M3     report='a'


  claude-sonnet-5      -> HTTP 200  7668 ms model_eff=glm-5.3        report='a'


## 5. Protocole pré-enregistré — gel des paramètres

**Avant le premier appel de mesure** : figer les paramètres ci-dessous. Toute modification en cours de route invalide le protocole.


In [5]:
PROTOCOL = {
    'seeds': SEEDS,
    'profiles': [('a', 'a'), ('a', A_PRIME), (A_PRIME, 'a'), (A_PRIME, A_PRIME)],
    'model_aliases': ['claude-haiku-4-5', 'claude-sonnet-5'],
    'conditions': ['visible', 'blind'],
    'temperature': 0.7,
    'max_tokens': 8,
    'scenarios': {
        'baseline': "règles + code visibles, LLM joue contre baseline conformiste",
        'homo_llm': 'LLM vs LLM (même alias), symétrique ou antisymétrique par seed',
        'hostile': "code PERTURBÉ : la règle 'boxed truthful o1' remplacée par 'boxed truthful o2'",
    },
    'hypotheses': {
        'H0': 'LLM aveugle <= baseline conformiste (payoff moyen) sur les 4 profils',
        'H1': 'LLM voyant > baseline conformiste et manipulabilité < baseline byzantine',
    },
    'metrics': ['payoff_row', 'payoff_col', 'sw', 'manipulability',
                'latency_ms', 'tokens_in', 'tokens_out', 'parse_errors'],
}

print('Protocole gelé :')
print(json.dumps(PROTOCOL, indent=2, ensure_ascii=False))


Protocole gelé :
{
  "seeds": [
    0,
    1,
    7,
    42
  ],
  "profiles": [
    [
      "a",
      "a"
    ],
    [
      "a",
      "a'"
    ],
    [
      "a'",
      "a"
    ],
    [
      "a'",
      "a'"
    ]
  ],
  "model_aliases": [
    "claude-haiku-4-5",
    "claude-sonnet-5"
  ],
  "conditions": [
    "visible",
    "blind"
  ],
  "temperature": 0.7,
  "max_tokens": 8,
  "scenarios": {
    "baseline": "règles + code visibles, LLM joue contre baseline conformiste",
    "homo_llm": "LLM vs LLM (même alias), symétrique ou antisymétrique par seed",
    "hostile": "code PERTURBÉ : la règle 'boxed truthful o1' remplacée par 'boxed truthful o2'"
  },
  "hypotheses": {
    "H0": "LLM aveugle <= baseline conformiste (payoff moyen) sur les 4 profils",
    "H1": "LLM voyant > baseline conformiste et manipulabilité < baseline byzantine"
  },
  "metrics": [
    "payoff_row",
    "payoff_col",
    "sw",
    "manipulability",
    "latency_ms",
    "tokens_in",
    "tokens_out",
    "p

## 6. Tournoi LLM × LLM / LLM × baseline (1 seed × 4 profils × 2 modèles × 2 conditions)

Pour chaque combinaison : appeler LLM-`row` (rapport sur type_row) puis LLM-`col` (rapport sur type_col, conditionné au rapport de l'adversaire `row`), passer les deux rapports à l'oracle, mesurer les payoff/SW/latence/tokens.

Le nombre total d'appels : 1 seed × 4 profils × 2 modèles × 2 conditions × 2 rôles = **32 appels**. À ~1-3 s chacun, c'est ~2-5 min de proxy. Le protocole prévoit 4 seeds, on documente le single-seed (pilote, pas benchmark formel).


In [6]:
def build_prompt(condition, type_self, opponent_report):
    if condition == 'visible':
        return PROMPT_VISIBLE_TEMPLATE.format(
            oracle_code=ORACLE_CODE_VISIBLE,
            type_self=type_self,
            opponent_report=opponent_report,
        )
    return PROMPT_BLIND_TEMPLATE.format(type_self=type_self, opponent_report=opponent_report)


results = []
t_total = time.time()
for condition in PROTOCOL['conditions']:
    for model_alias in PROTOCOL['model_aliases']:
        for tr, tc in PROTOCOL['profiles']:
            # row décide seul, puis col voit le rapport de row.
            prompt_row = build_prompt(condition, tr, opponent_report='?')
            res_row = call_llm(model_alias, prompt_row,
                                temperature=PROTOCOL['temperature'], seed=SEEDS[0])
            prompt_col = build_prompt(condition, tc, opponent_report=res_row['report'])
            res_col = call_llm(model_alias, prompt_col,
                                temperature=PROTOCOL['temperature'], seed=SEEDS[0])
            outcome = outcome_strict_mom(res_row['report'], res_col['report'])
            results.append({
                'condition': condition,
                'model_alias': model_alias,
                'type_row': tr, 'type_col': tc,
                'report_row': res_row['report'],
                'report_col': res_col['report'],
                'outcome': outcome,
                'payoff_row': payoff_row(outcome, tr),
                'payoff_col': payoff_col(outcome, tc),
                'sw': sw(outcome, tr, tc),
                'model_eff_row': res_row['model_effective'],
                'model_eff_col': res_col['model_effective'],
                'latency_row_ms': res_row['latency_ms'],
                'latency_col_ms': res_col['latency_ms'],
                'tokens_in': (res_row['usage'].get('prompt_tokens', 0)
                              + res_col['usage'].get('prompt_tokens', 0)),
                'tokens_out': (res_row['usage'].get('completion_tokens', 0)
                               + res_col['usage'].get('completion_tokens', 0)),
                'row_err': res_row.get('error'),
                'col_err': res_col.get('error'),
            })
dt_total = time.time() - t_total
print('\n=== Tournoi terminé en {:.1f}s ({} parties) ===\n'.format(dt_total, len(results)))
for r in results:
    print('[{cond:6}] {mod:15} ({tr},{tc}) rep=({rr},{rc}) out={o} u_r={ur} u_c={uc} SW={sw} lat=({lar},{lac})ms'.format(
        cond=r['condition'], mod=r['model_alias'], tr=r['type_row'], tc=r['type_col'],
        rr=r['report_row'], rc=r['report_col'], o=r['outcome'],
        ur=r['payoff_row'], uc=r['payoff_col'], sw=r['sw'],
        lar=r['latency_row_ms'], lac=r['latency_col_ms']))



=== Tournoi terminé en 131.8s (16 parties) ===

[visible] claude-haiku-4-5 (a,a) rep=(a,a) out=o2 u_r=4 u_c=0 SW=4 lat=(6117,2675)ms
[visible] claude-haiku-4-5 (a,a') rep=(a,a) out=o2 u_r=4 u_c=0 SW=4 lat=(1240,1329)ms
[visible] claude-haiku-4-5 (a',a) rep=(a,a) out=o2 u_r=5 u_c=0 SW=5 lat=(6058,990)ms
[visible] claude-haiku-4-5 (a',a') rep=(a,a) out=o2 u_r=5 u_c=0 SW=5 lat=(15664,4826)ms
[visible] claude-sonnet-5 (a,a) rep=(a,a) out=o2 u_r=4 u_c=0 SW=4 lat=(4084,3732)ms
[visible] claude-sonnet-5 (a,a') rep=(a,a) out=o2 u_r=4 u_c=0 SW=4 lat=(9746,2636)ms
[visible] claude-sonnet-5 (a',a) rep=(a,a) out=o2 u_r=5 u_c=0 SW=5 lat=(3526,3391)ms
[visible] claude-sonnet-5 (a',a') rep=(a,a) out=o2 u_r=5 u_c=0 SW=5 lat=(1564,1110)ms
[blind ] claude-haiku-4-5 (a,a) rep=(a,a) out=o2 u_r=4 u_c=0 SW=4 lat=(852,307)ms
[blind ] claude-haiku-4-5 (a,a') rep=(a,a) out=o2 u_r=4 u_c=0 SW=4 lat=(589,733)ms
[blind ] claude-haiku-4-5 (a',a) rep=(a,a) out=o2 u_r=5 u_c=0 SW=5 lat=(757,4738)ms
[blind ] claude-ha

## 7. Métriques agrégées + test H0/H1

**Manipulabilité** (par rôle) = `payoff(réel) - payoff(truthful)`. Pour le rôle `row`, `payoff_truthful` = payoff que `row` aurait eu si l'outcome était calculé pour le profil de TYPES (rapports = types). Si la manipulabilité est positive, l'agent a gagné à dévier (ou l'autre a dévier en sa faveur).

**Test H0** : payoff moyen LLM (condition `blind`) ≤ payoff moyen baseline conformiste.
**Test H1** : payoff moyen LLM (condition `visible`) > payoff moyen baseline conformiste ET manipulabilité moyenne < manipulabilité moyenne baseline byzantine.

Ces deux tests sont **observationnels** (single-seed), pas des tests statistiques formels — un pilote sérieux exigerait ≥4 seeds et un test de Wilcoxon ou équivalent. On le documente.


In [7]:
# Baselines de référence
baseline_conformist_payoffs = []
baseline_byzantine_payoffs = []
baseline_conformist_sws = []
baseline_byzantine_sws = []
for tr, tc in PROTOCOL['profiles']:
    g_c = run_baseline_game(BASELINES['conformist'], BASELINES['conformist'], tr, tc)
    g_b = run_baseline_game(BASELINES['byzantine'], BASELINES['byzantine'], tr, tc)
    baseline_conformist_payoffs.extend([g_c['payoff_row'], g_c['payoff_col']])
    baseline_byzantine_payoffs.extend([g_b['payoff_row'], g_b['payoff_col']])
    baseline_conformist_sws.append(g_c['sw'])
    baseline_byzantine_sws.append(g_b['sw'])

print('Baseline conformiste : payoffs moyens = {:.2f}, SW moyens = {:.2f}'.format(
    statistics.mean(baseline_conformist_payoffs), statistics.mean(baseline_conformist_sws)))
print('Baseline byzantine   : payoffs moyens = {:.2f}, SW moyens = {:.2f}'.format(
    statistics.mean(baseline_byzantine_payoffs), statistics.mean(baseline_byzantine_sws)))

# Manipulabilité par résultat : payoff réel - payoff si les deux rapportent leurs types
for r in results:
    pay_truth_row = payoff_row(outcome_strict_mom(r['type_row'], r['type_col']), r['type_row'])
    pay_truth_col = payoff_col(outcome_strict_mom(r['type_row'], r['type_col']), r['type_col'])
    r['manip_row'] = r['payoff_row'] - pay_truth_row
    r['manip_col'] = r['payoff_col'] - pay_truth_col

agg = {}
for r in results:
    key = (r['condition'], r['model_alias'])
    if key not in agg:
        agg[key] = {'payoffs': [], 'sw': [], 'manip': [], 'latencies': [],
                    'tokens_in': 0, 'tokens_out': 0}
    agg[key]['payoffs'].extend([r['payoff_row'], r['payoff_col']])
    agg[key]['sw'].append(r['sw'])
    agg[key]['manip'].extend([r['manip_row'], r['manip_col']])
    agg[key]['latencies'].extend([r['latency_row_ms'], r['latency_col_ms']])
    agg[key]['tokens_in'] += r['tokens_in']
    agg[key]['tokens_out'] += r['tokens_out']

print('\n=== Métriques agrégées par (condition, modèle) ===')
for key in sorted(agg.keys()):
    cond, mod = key
    a = agg[key]
    print('\n[{:6}] {:15} : n_appels={}'.format(cond, mod, len(a['latencies'])))
    print('  payoff moyen   : {:.2f}'.format(statistics.mean(a['payoffs'])))
    print('  SW moyen       : {:.2f}'.format(statistics.mean(a['sw'])))
    print('  manipulabilité : {:+.2f} (max={:+d}, min={:+d})'.format(
        statistics.mean(a['manip']), max(a['manip']), min(a['manip'])))
    print('  latence moy.   : {:.0f} ms'.format(statistics.mean(a['latencies'])))
    print('  tokens total   : in={}, out={}'.format(a['tokens_in'], a['tokens_out']))

# Test H0 / H1 (observationnel)
print('\n=== Test H0/H1 (observationnel, single-seed) ===')
ref_conf = statistics.mean(baseline_conformist_payoffs)
# Manipulabilité byzantine (rapport constant 'a' vs type vrai)
byz_games = [run_baseline_game(BASELINES['byzantine'], BASELINES['byzantine'], tr, tc)
             for tr, tc in PROTOCOL['profiles']]
ref_byz_manip = statistics.mean([
    g['payoff_row'] - payoff_row(outcome_strict_mom(g['type_row'], g['type_col']), g['type_row'])
    for g in byz_games
])
print('Référence conformiste (payoff moyen) : {:.2f}'.format(ref_conf))
print('Référence byzantine (manipulabilité)  : {:+.2f}'.format(ref_byz_manip))

for key in sorted(agg.keys()):
    cond, mod = key
    a = agg[key]
    pay = statistics.mean(a['payoffs'])
    manip = statistics.mean(a['manip'])
    if cond == 'blind':
        verdict = 'H0 TENU' if pay <= ref_conf else 'H0 INFIRMÉ'
    else:
        verdict = 'H1 TENU' if (pay > ref_conf and manip < ref_byz_manip) else 'H1 INFIRMÉ (ou partiel)'
    print('  [{:6}] {:15} : payoff={:.2f} manip={:+.2f}  → {}'.format(cond, mod, pay, manip, verdict))


Baseline conformiste : payoffs moyens = 2.75, SW moyens = 5.50
Baseline byzantine   : payoffs moyens = 2.25, SW moyens = 4.50

=== Métriques agrégées par (condition, modèle) ===

[blind ] claude-haiku-4-5 : n_appels=8
  payoff moyen   : 2.25
  SW moyen       : 4.50
  manipulabilité : -0.50 (max=+4, min=-6)
  latence moy.   : 1097 ms
  tokens total   : in=577, out=21

[blind ] claude-sonnet-5 : n_appels=8
  payoff moyen   : 2.25
  SW moyen       : 4.50
  manipulabilité : -0.50 (max=+4, min=-6)
  latence moy.   : 2978 ms
  tokens total   : in=1004, out=64

[visible] claude-haiku-4-5 : n_appels=8
  payoff moyen   : 2.25
  SW moyen       : 4.50
  manipulabilité : -0.50 (max=+4, min=-6)
  latence moy.   : 4862 ms
  tokens total   : in=496, out=20

[visible] claude-sonnet-5 : n_appels=8
  payoff moyen   : 2.25
  SW moyen       : 4.50
  manipulabilité : -0.50 (max=+4, min=-6)
  latence moy.   : 3724 ms
  tokens total   : in=3436, out=64

=== Test H0/H1 (observationnel, single-seed) ===
Référe

## 8. Scénario hostile — permutation d'une règle

Le scénario hostile vérifie que **l'avantage LLM disparaît** quand le code que le LLM « voit » est **faux**. On remplace la règle `boxed truthful = o1` par `boxed truthful = o2` : tout LLM qui s'est fié au code va dévier inutilement.

Le coût LLM est neutralisé (on ne compte que les décisions, pas les USD) — l'hostilité est sur la **règle**, pas sur le budget.


In [8]:
# ORACLE_CODE_HOSTILE : la règle boxed truthful = o2 au lieu de o1
ORACLE_CODE_HOSTILE = (
    'Le mécanisme M est le suivant (code Python -- ATTENTION, ce code est PERTURBÉ) :\n'
    '    U_ROW = {\'o1\': {\'a\': 1, \'a' + chr(39) + '\': 3}, \'o2\': {\'a\': 4, \'a' + chr(39) + '\': 5},\n'
    '             \'o3\': {\'a\': 0, \'a' + chr(39) + '\': 0}, \'o4\': {\'a\': 3, \'a' + chr(39) + '\': 0}}\n'
    '    U_COL = {\'o1\': {\'a\': 1, \'a' + chr(39) + '\': 4}, \'o2\': {\'a\': 0, \'a' + chr(39) + '\': 0},\n'
    '             \'o3\': {\'a\': 3, \'a' + chr(39) + '\': 6}, \'o4\': {\'a\': 0, \'a' + chr(39) + '\': 0}}\n'
    '    def outcome(rep_r, rep_c):\n'
    '        if rep_r == \'a' + chr(39) + '\' and rep_c == \'a' + chr(39) + '\': return \'o2\'   # PERTURBATION\n'
    '        return max([\'o1\',\'o2\',\'o3\',\'o4\'],\n'
    '                   key=lambda o: U_ROW[o][rep_r] + U_COL[o][rep_c])\n'
    '    def payoff_row(o, tr): return U_ROW[o][tr]\n'
    '    def payoff_col(o, tc): return U_COL[o][tc]\n'
)

PROMPT_HOSTILE_TEMPLATE = (
    "Tu joues au jeu Othman-Sandholm.\n"
    "Règles : tu as un type privé parmi {{a, a_prime}}. Tu choisis un rapport parmi {{a, a_prime}}. "
    "Le mécanisme produit un outcome o1..o4. Ton utility dépend de l'outcome et de ton type.\n"
    "\n"
    "{oracle_code}\n"
    "\n"
    "Ton type (privé) : {type_self}\n"
    "Le rapport de l'adversaire est : {opponent_report}\n"
    "Réponds UNIQUEMENT par 'a' ou par \"a'\" (avec l'apostrophe). Aucun autre texte.\n"
)


def outcome_strict_mom_HOSTILE(rep_r, rep_c):
    """Même mécanique, mais boxed truthful = o2."""
    if rep_r == A_PRIME and rep_c == A_PRIME:
        return 'o2'
    return max(OUTCOMES, key=lambda o: U_ROW[o][rep_r] + U_COL[o][rep_c])


hostile_results = []
for model_alias in PROTOCOL['model_aliases']:
    for tr, tc in PROTOCOL['profiles']:
        prompt_row = PROMPT_HOSTILE_TEMPLATE.format(
            oracle_code=ORACLE_CODE_HOSTILE, type_self=tr, opponent_report='?')
        res_row = call_llm(model_alias, prompt_row,
                            temperature=PROTOCOL['temperature'], seed=SEEDS[0])
        prompt_col = PROMPT_HOSTILE_TEMPLATE.format(
            oracle_code=ORACLE_CODE_HOSTILE, type_self=tc, opponent_report=res_row['report'])
        res_col = call_llm(model_alias, prompt_col,
                            temperature=PROTOCOL['temperature'], seed=SEEDS[0])
        # IMPORTANT : appliquer l'oracle RÉEL (pas le code hostile vu par le LLM)
        outcome = outcome_strict_mom_HOSTILE(res_row['report'], res_col['report'])
        hostile_results.append({
            'model_alias': model_alias,
            'type_row': tr, 'type_col': tc,
            'report_row': res_row['report'],
            'report_col': res_col['report'],
            'outcome': outcome,
            'payoff_row': payoff_row(outcome, tr),
            'payoff_col': payoff_col(outcome, tc),
            'sw': sw(outcome, tr, tc),
        })

print('=== Scénario hostile (oracle RÉEL = boxed truthful o2) ===')
for h in hostile_results:
    print('  {:15} ({},{}) rep=({},{}) out={} u_r={} u_c={} SW={}'.format(
        h['model_alias'], h['type_row'], h['type_col'],
        h['report_row'], h['report_col'], h['outcome'],
        h['payoff_row'], h['payoff_col'], h['sw']))

hostile_payoffs = [p for h in hostile_results for p in [h['payoff_row'], h['payoff_col']]]
visible_payoffs = []
for key in [('visible', 'claude-haiku-4-5'), ('visible', 'claude-sonnet-5')]:
    visible_payoffs.extend(agg[key]['payoffs'])
print('\nPayoff moyen (hostile) : {:.2f}'.format(statistics.mean(hostile_payoffs)))
print('Payoff moyen (visible, oracle correct) : {:.2f}'.format(statistics.mean(visible_payoffs)))
print('=> L\'écart payoff entre oracle correct et oracle hostile mesure l\'avantage LLM.')


=== Scénario hostile (oracle RÉEL = boxed truthful o2) ===
  claude-haiku-4-5 (a,a) rep=(a,a) out=o2 u_r=4 u_c=0 SW=4
  claude-haiku-4-5 (a,a') rep=(a,a) out=o2 u_r=4 u_c=0 SW=4
  claude-haiku-4-5 (a',a) rep=(a,a) out=o2 u_r=5 u_c=0 SW=5
  claude-haiku-4-5 (a',a') rep=(a,a) out=o2 u_r=5 u_c=0 SW=5
  claude-sonnet-5 (a,a) rep=(a,a) out=o2 u_r=4 u_c=0 SW=4
  claude-sonnet-5 (a,a') rep=(a,a) out=o2 u_r=4 u_c=0 SW=4
  claude-sonnet-5 (a',a) rep=(a,a) out=o2 u_r=5 u_c=0 SW=5
  claude-sonnet-5 (a',a') rep=(a,a) out=o2 u_r=5 u_c=0 SW=5

Payoff moyen (hostile) : 2.25
Payoff moyen (visible, oracle correct) : 2.25
=> L'écart payoff entre oracle correct et oracle hostile mesure l'avantage LLM.


## 9. Exercices (C.1 — pas d'erreur volontaire)

Les trois exercices ci-dessous suivent la règle **C.1** : stubs corrects (`return None`, `# TODO`), pas de `raise NotImplementedError`. Le notebook s'exécute de bout en bout même exercices non complétés.


In [9]:
# === Exercice 1 : étalonner le protocole avec les baselines seules ===
def exercice1_etalonnage_baselines():
    """
    Vérifier que la Caractéristique 2 tient pour tous les profils byzantin :
    il existe un outcome avec SW > SW(o1) pour les profils (a, a), (a, a'), (a', a).
    """
    # TODO etudiant : implémenter et retourner un dict {"all_pass": bool, "details": [...]}
    return {'all_pass': None, 'details': None}  # stub C.1 conforme

print('Exercice 1 (étalonnage) :', exercice1_etalonnage_baselines())

# === Exercice 2 : reproductibilité manuelle ===
def exercice2_reproductibilite_manuelle():
    """
    Choisir un des résultats LLM ci-dessus, imprimer le prompt exact,
    et retourner la décision que l'étudiant prendrait (sans voir la décision LLM).
    """
    # TODO etudiant : choisir un (model_alias, type_row, condition), imprimer le prompt,
    # et retourner {"prompt": ..., "decision": ..., "justification": ...}
    return {'prompt': None, 'decision': None, 'justification': None}

print('Exercice 2 (reproductibilité) :', exercice2_reproductibilite_manuelle())

# === Exercice 3 : scénario hostile — effondrement attendu ===
def exercice3_effondrement_hostile():
    """
    Calculer l'écart payoff moyen entre visible-correct et hostile-pour-le-LLM.
    """
    # TODO etudiant : agréger payoff moyen sur visible-correct et hostile,
    # retourner {"ecart": float, "verdict": "..."}
    return {'ecart': None, 'verdict': None}

print('Exercice 3 (hostile) :', exercice3_effondrement_hostile())


Exercice 1 (étalonnage) : {'all_pass': None, 'details': None}
Exercice 2 (reproductibilité) : {'prompt': None, 'decision': None, 'justification': None}
Exercice 3 (hostile) : {'ecart': None, 'verdict': None}


## 10. Résumé et limites

**Ce qui a été livré** :
- Pilote de **2 familles LLM hétérogènes** (via proxy claudish) jouant Othman–Sandholm Proposition 6.
- **2 baselines scriptées** (conformiste DSIC + byzantin uniforme) sur les mêmes 4 profils.
- **2 conditions** (voit / aveugle) × **2 modèles** × **4 profils** × **2 rôles** = 32 appels (single-seed).
- **Scénario hostile** : permutation de la règle boxed truthful + coût neutralisé, pour vérifier que l'avantage LLM disparaît.
- **3 exercices C.1** : étalonnage baselines / reproductibilité manuelle / effondrement hostile.

**Limites explicites** :
- **Single-seed** : 4 seeds prévus au protocole, 1 seul exécuté. Les verdicts H0/H1 sont observationnels, non statistiques.
- **Coût non-monétisé** : tokens mesurés, USD non estimés (pas de grille tarifaire publique pour les modèles distants). La métrique `tokens_in/out` est conservée pour extension future.
- **Hétérogénéité effective** : les deux modèles sont mappés en interne à deux backends distincts (vérifié par le champ `model` dans la réponse proxy), mais les alias de requête passent par la même URL. Le pilote caractérise l'hétérogénéité backend, pas l'hétérogénéité d'API.
- **Pas de claim Löb/FairBot** : le pilote manipule l'utilité, pas la vérité gödelienne.

**Voir aussi** :
- Issue #15399 (pilote), Epic #15397 (C-documentaire)
- PR Lean #12343 (Proposition 6 formalisée par `decide`)
- `GameTheory-16-MechanismDesign.ipynb` §4.6 (notebook owner du mécanisme)
